# 04 Statistical Analysis

This notebook performs statistical analysis on the cleaned Swiggy dataset:
1. Descriptive statistics grouped by City and Category
2. Average Price by Category and City
3. Hypothesis testing — ANOVA (Rating by City, Rating by Category)
4. Spearman correlation — Price vs Rating

All results are saved to `reports/stats/`.

In [4]:
from pathlib import Path

import pandas as pd
from scipy import stats

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name.strip() == 'notebooks' else current_dir
STATS_PATH = PROJECT_ROOT / 'reports/stats'
STATS_PATH.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05

In [5]:
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/swiggy_cleaned.csv'
df = pd.read_csv(PROCESSED_PATH, parse_dates=['Order Date'])
print('Loaded:', df.shape)
df.head()

Loaded: (91788, 11)


,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,is_price_outlier
0,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25.0,False
1,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48.0,False
2,Karnataka,Bengaluru,2025-01-21,Srinidhi Sagar Deluxe,Kengeri,Recommended,Garlic Naan,98.0,4.0,34.0,False
3,Karnataka,Bengaluru,2025-05-02,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Panneer Butter Masala,241.0,4.4,29.0,False
4,Karnataka,Bengaluru,2025-07-30,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Dal Tadka,195.0,4.9,51.0,False


## 1. Descriptive Statistics — Grouped by City

In [6]:
grouped_city = (
    df.groupby('City')[['Price (INR)', 'Rating', 'Rating Count']]
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(2)
)
grouped_city.columns = ['_'.join(col) for col in grouped_city.columns]
grouped_city = grouped_city.reset_index()
grouped_city.to_csv(STATS_PATH / 'grouped_stats_by_city.csv', index=False)
print('Grouped stats by City saved.')
grouped_city

Grouped stats by City saved.


,City,Price (INR)_mean,Price (INR)_median,Price (INR)_std,Price (INR)_min,Price (INR)_max,Rating_mean,Rating_median,Rating_std,Rating_min,Rating_max,Rating Count_mean,Rating Count_median,Rating Count_std,Rating Count_min,Rating Count_max
0,Agartala,198.06,182.00,149.09,0.95,2199.00,4.25,4.40,0.57,2.0,5.0,69.27,18.0,138.10,1.0,996.0
1,Ahmedabad,256.77,226.00,176.98,0.95,2599.00,4.38,4.50,0.51,2.0,5.0,35.32,6.0,93.04,1.0,994.0
2,Aizawl,238.97,208.00,170.64,0.95,2199.00,4.43,4.60,0.53,2.1,5.0,11.15,5.0,17.17,1.0,138.0
3,Bengaluru,255.56,229.00,198.69,0.95,3200.00,4.30,4.40,0.53,1.5,5.0,41.69,10.0,95.02,1.0,994.0
4,Bhubaneswar,220.87,212.00,140.09,0.95,1929.00,4.35,4.40,0.51,2.0,5.0,87.09,24.0,160.97,1.0,998.0
5,Chandigarh,265.74,229.52,195.89,0.95,2199.00,4.31,4.40,0.51,2.0,5.0,53.58,11.0,118.86,1.0,989.0
6,Chennai,246.79,209.00,198.91,0.95,3899.00,4.35,4.40,0.51,1.5,5.0,60.65,12.0,134.42,1.0,992.0
7,Dehradun,240.28,209.00,155.92,0.95,1300.38,4.21,4.30,0.58,2.0,5.0,40.58,10.0,89.66,1.0,997.0
8,Gangtok,211.47,189.88,156.29,0.95,1783.65,4.16,4.30,0.64,2.1,5.0,13.52,6.0,22.68,1.0,160.0
9,Gurgaon,273.38,240.00,201.75,0.95,3200.00,4.33,4.40,0.54,1.5,5.0,46.74,9.0,116.64,1.0,972.0


## 2. Descriptive Statistics — Grouped by Category

In [7]:
grouped_category = (
    df.groupby('Category')[['Price (INR)', 'Rating', 'Rating Count']]
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(2)
)
grouped_category.columns = ['_'.join(col) for col in grouped_category.columns]
grouped_category = grouped_category.reset_index()
grouped_category.to_csv(STATS_PATH / 'grouped_stats_by_category.csv', index=False)
print('Grouped stats by Category saved.')
grouped_category

Grouped stats by Category saved.


,Category,Price (INR)_mean,Price (INR)_median,Price (INR)_std,Price (INR)_min,Price (INR)_max,Rating_mean,Rating_median,Rating_std,Rating_min,Rating_max,Rating Count_mean,Rating Count_median,Rating Count_std,Rating Count_min,Rating Count_max
0,Desi Ghee Sweets,580.87,600.0,263.22,245.0,1180.0,4.75,4.80,0.24,4.2,5.0,27.87,23.0,22.40,4.0,83.0
1,( Chinese),115.00,120.0,37.75,75.0,150.0,2.73,3.00,0.64,2.0,3.2,3.00,3.0,0.00,3.0,3.0
2,(Sides),75.00,75.0,NaN,75.0,75.0,4.00,4.00,NaN,4.0,4.0,5.00,5.0,NaN,5.0,5.0
3,*New* High-Protein Rice Bowls,199.00,199.0,0.00,199.0,199.0,4.52,4.60,0.42,3.8,5.0,3.83,3.0,3.06,2.0,10.0
4,*New* High-Protein Roti Meals,299.00,299.0,0.00,299.0,299.0,3.75,3.75,0.07,3.7,3.8,5.00,5.0,2.83,3.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3777,Zafrani,150.00,150.0,NaN,150.0,150.0,4.10,4.10,NaN,4.1,4.1,10.00,10.0,NaN,10.0,10.0
3778,Zayka -Mutton,127.00,127.0,NaN,127.0,127.0,4.50,4.50,NaN,4.5,4.5,70.00,70.0,NaN,70.0,70.0
3779,Zerpian,360.00,320.0,105.83,280.0,480.0,4.50,4.50,0.10,4.4,4.6,14.67,5.0,17.62,4.0,35.0
3780,[750Gram] Dum Biryani,779.00,779.0,NaN,779.0,779.0,3.90,3.90,NaN,3.9,3.9,1.00,1.0,NaN,1.0,1.0


## 3. Average Price by Category

In [8]:
avg_price_category = (
    df.groupby('Category')['Price (INR)']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'Price (INR)': 'Avg Price (INR)'})
    .sort_values('Avg Price (INR)', ascending=False)
)
avg_price_category.to_csv(STATS_PATH / 'avg_price_by_category.csv', index=False)
print('Average price by Category saved.')
avg_price_category

Average price by Category saved.


,Category,Avg Price (INR)
309,Biryani Bucket [Serves 4-6],2349.00
1206,Family Meal Box (serves 4 To 6),1999.00
1505,Hampers & Gift Boxes,1988.00
2218,Monster Pizza (24 Inches),1979.00
283,Big Big Pizza,1919.24
...,...,...
2931,Rotiyaan,15.00
2595,Pav,10.00
2566,Party Item,10.00
2599,Pav Bhaji Special,10.00


## 4. Average Price by City

In [9]:
avg_price_city = (
    df.groupby('City')['Price (INR)']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'Price (INR)': 'Avg Price (INR)'})
    .sort_values('Avg Price (INR)', ascending=False)
)
avg_price_city.to_csv(STATS_PATH / 'avg_price_by_city.csv', index=False)
print('Average price by City saved.')
avg_price_city

Average price by City saved.


,City,Avg Price (INR)
21,Panaji,288.79
19,Mumbai,274.74
9,Gurgaon,273.38
18,Lucknow,270.30
25,Shillong,268.03
11,Hyderabad,266.00
5,Chandigarh,265.74
20,New Delhi,261.44
1,Ahmedabad,256.77
3,Bengaluru,255.56


## 5. Hypothesis Testing

### 5a. ANOVA — Does Rating differ significantly across Cities?

In [10]:
city_groups = [group['Rating'].dropna().values for _, group in df.groupby('City')]
f_stat_city, p_val_city = stats.f_oneway(*city_groups)

print('ANOVA — Rating by City')
print(f'  F-statistic : {f_stat_city:.4f}')
print(f'  p-value     : {p_val_city:.4f}')
print(f'  Result      : {"Significant difference" if p_val_city < ALPHA else "No significant difference"} at α = {ALPHA}')

ANOVA — Rating by City
  F-statistic : 62.9676
  p-value     : 0.0000
  Result      : Significant difference at α = 0.05


### 5b. ANOVA — Does Rating differ significantly across Categories?

In [11]:
category_groups = [group['Rating'].dropna().values for _, group in df.groupby('Category')]
f_stat_cat, p_val_cat = stats.f_oneway(*category_groups)

print('ANOVA — Rating by Category')
print(f'  F-statistic : {f_stat_cat:.4f}')
print(f'  p-value     : {p_val_cat:.4f}')
print(f'  Result      : {"Significant difference" if p_val_cat < ALPHA else "No significant difference"} at α = {ALPHA}')

ANOVA — Rating by Category
  F-statistic : 3.1207
  p-value     : 0.0000
  Result      : Significant difference at α = 0.05


### 5c. Spearman Correlation — Price vs Rating

In [12]:
corr_data = df[['Price (INR)', 'Rating']].dropna()
spearman_corr, spearman_p = stats.spearmanr(corr_data['Price (INR)'], corr_data['Rating'])

print('Spearman Correlation — Price vs Rating')
print(f'  Correlation : {spearman_corr:.4f}')
print(f'  p-value     : {spearman_p:.4f}')
print(f'  Result      : {"Significant correlation" if spearman_p < ALPHA else "No significant correlation"} at α = {ALPHA}')

Spearman Correlation — Price vs Rating
  Correlation : 0.0286
  p-value     : 0.0000
  Result      : Significant correlation at α = 0.05


## 6. Save Hypothesis Test Results

In [13]:
hypothesis_results = pd.DataFrame([
    {
        'Test'        : 'ANOVA',
        'Description' : 'Rating by City',
        'F-Statistic' : round(f_stat_city, 4),
        'p-value'     : round(p_val_city, 4),
        'Significant' : p_val_city < ALPHA
    },
    {
        'Test'        : 'ANOVA',
        'Description' : 'Rating by Category',
        'F-Statistic' : round(f_stat_cat, 4),
        'p-value'     : round(p_val_cat, 4),
        'Significant' : p_val_cat < ALPHA
    },
    {
        'Test'        : 'Spearman Correlation',
        'Description' : 'Price vs Rating',
        'F-Statistic' : round(spearman_corr, 4),
        'p-value'     : round(spearman_p, 4),
        'Significant' : spearman_p < ALPHA
    }
])
hypothesis_results.to_csv(STATS_PATH / 'hypothesis_test_results.csv', index=False)
print('Hypothesis test results saved.')
hypothesis_results

Hypothesis test results saved.


,Test,Description,F-Statistic,p-value,Significant
0,ANOVA,Rating by City,62.9676,0.0,True
1,ANOVA,Rating by Category,3.1207,0.0,True
2,Spearman Correlation,Price vs Rating,0.0286,0.0,True
